In [2]:
# ============================================================
# SILVER → GOLD
# Enterprise Finance Medallion Architecture
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================================
# READ SILVER TABLE
# ============================================================

silver_table_path = "abfss://MariaMonedero@onelake.dfs.fabric.microsoft.com/lh_enterprise_silver.Lakehouse/Tables/dbo/silver_fact_operations"

df = spark.read.format("delta").load(silver_table_path)

display(df)

StatementMeta(, 617aa672-a0fe-47a5-a020-aff557131797, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 96a48e74-d0c4-4d05-93b3-096b895ea543)

In [3]:
# ============================================================
# DIM DATE
# ============================================================

dim_date = ( 
    df
    .filter(col("Date").isNotNull())
    .select(
        col("Date"),
        year(col("Date")).alias("Year"),
        month(col("Date")).alias("Month"),
        date_format(col("Date"), "MMM").alias("Month_Name"),
        quarter(col("Date")).alias("Quarter")
    )
    .distinct()
    .orderBy("Date")
)

display(dim_date)

StatementMeta(, 617aa672-a0fe-47a5-a020-aff557131797, 5, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, f9d091de-552b-4b37-83ff-3368f3738716)

In [4]:
# ============================================================
# DIM DEPARTMENT
# ============================================================

from pyspark.sql.window import Window

dim_department = (
    df.select("Department")
    .distinct()
)

window_spec = Window.orderBy("Department")

dim_department = dim_department.withColumn(
    "Department_ID",
    row_number().over(window_spec)
)

display(dim_department)

StatementMeta(, 8c73639f-9192-4ab0-94b5-210ba278b2b9, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 569aad6f-98df-49c8-bb5a-6170d62178cc)

In [5]:
# ============================================================
# DIM BUSINESS UNIT
# ============================================================

window_spec = Window.orderBy("Business_Unit")

dim_business_unit = (
    df.select("Business_Unit")
    .distinct()
    .withColumn(
        "Business_Unit_ID",
        row_number().over(window_spec)
    )
)

display(dim_business_unit)

StatementMeta(, 8c73639f-9192-4ab0-94b5-210ba278b2b9, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05e8a72b-bf28-4b5d-ac8e-301623d1ee89)

In [6]:
# ============================================================
# DIM RISK
# ============================================================

window_spec = Window.orderBy("Risk_Level")

dim_risk = (
    df.select("Risk_Level")
    .distinct()
    .withColumn(
        "Risk_ID",
        row_number().over(window_spec)
    )
)

display(dim_risk)

StatementMeta(, 8c73639f-9192-4ab0-94b5-210ba278b2b9, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ab498128-b18b-46c1-9bc1-307538c3279f)

In [7]:
# ============================================================
# DIM PROJECT STATUS
# ============================================================

window_spec = Window.orderBy("Project_Status")

dim_project_status = (
    df.select("Project_Status")
    .distinct()
    .withColumn(
        "Project_Status_ID",
        row_number().over(window_spec)
    )
)

display(dim_project_status)

StatementMeta(, 8c73639f-9192-4ab0-94b5-210ba278b2b9, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7856920a-3b5e-4b51-9e1d-ef9652a2c90f)

In [ ]:
df = df.filter(col("Date").isNotNull())

In [8]:
# ============================================================
# FACT TABLE WITH DIMENSION KEYS
# ============================================================

fact_df = (
    df
    .join(dim_department, on="Department", how="left")
    .join(dim_business_unit, on="Business_Unit", how="left")
    .join(dim_risk, on="Risk_Level", how="left")
    .join(dim_project_status, on="Project_Status", how="left")
)

display(fact_df)

StatementMeta(, 8c73639f-9192-4ab0-94b5-210ba278b2b9, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, caad2a19-4cb5-4c75-b569-7270260d8a89)

In [9]:
# ============================================================
# SAVE GOLD TABLES
# ============================================================

fact_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_fact_operations")

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_date")

dim_department.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_department")

dim_business_unit.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_business_unit")

dim_risk.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_risk")

dim_project_status.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_project_status")

print("========================================")
print("GOLD LAYER CREATED SUCCESSFULLY")
print("========================================")
print("Tables created:")
print("- gold_fact_operations")
print("- gold_dim_date")
print("- gold_dim_department")
print("- gold_dim_business_unit")
print("- gold_dim_risk")
print("- gold_dim_project_status")
print(f"Fact rows: {fact_df.count()}")

StatementMeta(, 8c73639f-9192-4ab0-94b5-210ba278b2b9, 11, Finished, Available, Finished, True)

GOLD LAYER CREATED SUCCESSFULLY
Tables created:
- gold_fact_operations
- gold_dim_date
- gold_dim_department
- gold_dim_business_unit
- gold_dim_risk
- gold_dim_project_status
Fact rows: 28777
